# STREAM velocity embedding stream plots

This notebook loads the trained standard CFM, FiLM STREAM, and cross-attention STREAM checkpoints, predicts velocities for a sampled set of cells, and saves one `scvelo.pl.velocity_embedding_stream` plot per model variant.

The notebook expects the STREAM outputs to exist under `outputs/stream/` and the full-data UMAP coordinates from the EDA workflow under `outputs/jax_adata_eda/`.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
SRC_DIR = PROJECT_ROOT / "src"
USER_SITE = Path.home() / ".local" / "lib" / "python3.11" / "site-packages"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
if USER_SITE.exists() and str(USER_SITE) not in sys.path:
    sys.path.append(str(USER_SITE))

import gc
import torch
from stream_model.config import StreamConfig
from stream_model.scvelo_plotting import (
    add_velocity_layer,
    order_days_for_plot,
    plot_velocity_stream,
    predict_variant_velocity,
    sample_model_adata,
)

CONFIG_PATH = PROJECT_ROOT / "configs" / "stream_mouse_dev.yaml"
cfg = StreamConfig.from_yaml(CONFIG_PATH)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device}")

In [ ]:
N_PLOT_CELLS = int(os.environ.get("STREAM_SCVELO_CELLS", "2500"))
PREDICT_BATCH_SIZE = int(os.environ.get("STREAM_SCVELO_BATCH", "32"))
SEED = int(os.environ.get("STREAM_SCVELO_SEED", "1337"))
OUT_DIR = cfg.out_dir / "scvelo_stream"
OUT_DIR.mkdir(parents=True, exist_ok=True)

adata = sample_model_adata(cfg, n_cells=N_PLOT_CELLS, seed=SEED)
adata = order_days_for_plot(adata)
print(adata)
print(adata.obs[["day", "source_file"]].head())

In [ ]:
variants = ["standard_cfm", "film", "cross_attention"]
plot_paths = {}

for variant in variants:
    print(f"Predicting velocities for {variant}")
    velocity = predict_variant_velocity(
        adata,
        cfg,
        variant=variant,
        device=device,
        batch_size=PREDICT_BATCH_SIZE,
    )
    variant_adata = add_velocity_layer(adata, velocity, vkey="velocity")
    output_path = OUT_DIR / f"velocity_embedding_stream_{variant}.png"
    print(f"Plotting {output_path}")
    plot_velocity_stream(
        variant_adata,
        output_path=output_path,
        vkey="velocity",
        color="day",
        n_neighbors=30,
    )
    plot_paths[variant] = output_path

plot_paths